# Adaptive Boss — TRL (GRPO) Training

**Meta × Scaler OpenEnv Hackathon — Round 2 | Bangalore, April 25–26 2026**

This notebook trains the boss policy on the [`AdaptiveBossEnv`](https://github.com/) using
Hugging Face TRL's `GRPOTrainer` driving a tiny GPT-2 (~100K params) as the policy.

Pipeline:
1. Each prompt = current 13-dim env state encoded as a short token sequence
2. Each completion = single token in `{L, R, M, D}` → boss action 0..3
3. Reward = `AdaptiveBossEnv.step(decoded_action).reward`

**Note:** the production v7 boss model was trained with custom PyTorch PPO
(`train.py` + `rl/trainer.py`, 10 000 ep, 91% smoothed WR). This TRL pipeline is
the reproducible HF-stack alternative judges can run end-to-end here in Colab.
Both pipelines train against the same env, same reward, same action space.

## 1. Setup

In [ ]:
# Install pinned versions known to work with this repo's train_trl.py
!pip install -q "trl>=1.2.0,<2.0" "transformers>=5.2.0" accelerate datasets tokenizers matplotlib numpy

In [ ]:
# Clone the repo (Meta × Scaler OpenEnv Hackathon submission, Round 2)
import os
REPO_URL = os.environ.get("ADAPTIVE_BOSS_REPO", "https://github.com/heheDixo/Adaptive-Boss-RL.git")
!git clone $REPO_URL
%cd Adaptive-Boss-RL/adaptive_boss

## 2. Train

Default: 1500 states × 3 epochs × 4 generations per prompt ≈ 18 000 reward calls.
On a Colab GPU this should finish in 5–10 minutes.

In [ ]:
!python train_trl.py \
    --n_states 1500 \
    --epochs 3 \
    --num_generations 4 \
    --per_device_batch_size 16 \
    --eval_episodes 100 \
    --log_path logs/trl_training_log.json \
    --model_path models/boss_policy_trl

## 3. Plot training curves

In [ ]:
!python generate_trl_plot.py

from IPython.display import Image
Image('logs/trl_training_curve.png')

## 4. Compare to the production custom-PPO run

The repo ships a 3-panel `logs/training_curve.png` from the production custom-PPO
run (10 000 ep, 91% smoothed WR). Side-by-side with the TRL curve above, judges
can see both pipelines train the same env from scratch and the env produces
monotonically improving reward signal under either trainer.

In [ ]:
from IPython.display import Image
Image('logs/training_curve.png')

## 5. Next steps

- Inspect `models/boss_policy_trl/` — the saved tiny GPT-2 + tokenizer.
- Inspect `logs/trl_training_log_eval.json` — argmax win rate, draw rate, mean
  per-episode reward, and action distribution under the trained policy.
- Run the live Pygame demo locally (`python play.py`) to see the production
  custom-PPO boss in a 3-mode split-screen — trained / untrained / human.